# NB96 - Geometry-lever ablation @ 2k images (diagnostic)

**Pipeline position:** after NB95 confirms the lane head CAN learn on a
tiny set, this finds the smallest config change that makes `IoU(lane)`
and `[lane-geom]` move at a representative 2k-image scale - the config to
carry into the post-fix full run (NB97+).

| row | lane-weights | diff-clamp | match | question |
|---|---|---|---|---|
| A | cut5x | 100  | hungarian | reproduce the NB94 frozen baseline (control) |
| B | clrkd | none | hungarian | does strong, UNCLAMPED geometry supervision move IoU? |
| C | clrkd | none | dynamic_k | does DENSER matching (more positive priors) help geometry? |

3 epochs/row, batch 32. Watch each row's `IoU(lane)` slope and whether
`[lane-geom]` means move. The winning row = the full-run config.


### Cell 1: Mount + env

In [1]:
import os, sys, subprocess
from pathlib import Path
from google.colab import drive
os.environ['PYTHONIOENCODING'] = 'utf-8'
if not Path('/content/drive').exists(): drive.mount('/content/drive', force_remount=False)
REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT): raise FileNotFoundError(f'Missing {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path: sys.path.insert(0, REPO_ROOT)
try:
    import scipy, mmcv  # noqa
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'mmcv', 'scipy'])
print('[ok] env ready')


Mounted at /content/drive
[ok] env ready


### Cell 2: Prep 2k-image subset (2000 train + 500 val)

In [2]:
import sys, subprocess
from pathlib import Path
PREP = Path('stage2/rmt_ppad_migration/P8_train/scripts/prepare_bdd_subset.py')
ROOT = Path('/content/bdd_subset_2k')
if not PREP.exists(): raise FileNotFoundError(f'prep missing: {PREP} (sync Drive)')
if (ROOT/'prep_summary.json').exists():
    print(f'[ok] 2k subset present at {ROOT}; skipping')
else:
    cmd = [sys.executable, '-u', str(PREP), '--out-root', str(ROOT),
           '--n-train', '2000', '--n-val', '500', '--seed', '96']
    print('  cmd:', ' '.join(cmd), flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout: print(line, end='', flush=True)
    if proc.wait() != 0: raise RuntimeError('prep failed; see output above')
for lab, d in (('img/train', ROOT/'images/train2017'), ('img/val', ROOT/'images/val2017'),
               ('lt/train', ROOT/'lane_targets/train2017'), ('lt/val', ROOT/'lane_targets/val2017')):
    print(f'  {lab:10s}', sum(1 for _ in d.iterdir()) if d.exists() else 0)


  cmd: /usr/bin/python3 -u stage2/rmt_ppad_migration/P8_train/scripts/prepare_bdd_subset.py --out-root /content/bdd_subset_2k --n-train 2000 --n-val 500 --seed 96
  curve_tar   -> /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar
  labels_zip  -> /content/drive/MyDrive/EcoCAR/downloads/rmt_ppad_weights/BDD_detection_labels.zip
  lane_tar    -> /content/drive/MyDrive/EcoCAR/datasets/lane_targets_clr_v1_polyline.tar.gz
[prep] step 1: extract curve tarball
  extracting bdd100k_clrkd_curve.tar -> /content/bdd_curve_scratch
/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage2/rmt_ppad_migration/P8_train/scripts/prepare_bdd_subset.py:72: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(dest)
  done in 92.7s
  curve images: train=/content/bdd_curve_scratch/images/train (70000 jpgs), val=/content/bdd_curve_scratch/images/val (10000 jpgs)
[p

### Cell 3: Helper - `run_row(name, weights, clamp, match)`

In [3]:
import os, sys, subprocess, shutil
from pathlib import Path
from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'; os.makedirs(LOG_DIR, exist_ok=True)
YAML_MODEL = Path(REPO_ROOT)/'stage2/rmt_ppad_migration/vendor/RMT-PPAD/ultralytics/cfg/models/mt-detr/rtdetr-l_bdd_clr_lane.yaml'
YAML_DATA  = Path(REPO_ROOT)/'stage2/rmt_ppad_migration/vendor/RMT-PPAD/ultralytics/cfg/datasets/BDD_lane_only_2k.yaml'
TRAIN = 'stage2/rmt_ppad_migration/P8_train/scripts/train_lane_only.py'

def run_row(name, weights, clamp, match, epochs=3, batch=32):
    rd = Path('/content/runs')/name
    if rd.exists(): shutil.rmtree(rd, ignore_errors=True)
    cmd = [sys.executable, '-u', TRAIN, '--mode', 'full', '--name', name,
           '--project', '/content/runs', '--model-yaml', str(YAML_MODEL),
           '--data-yaml', str(YAML_DATA), '--device', '0', '--workers', '8',
           '--save-period', '50', '--batch', str(batch), '--epochs', str(epochs),
           '--lr0', '4e-4', '--lane-match', match, '--lane-weights', weights,
           '--diff-clamp', clamp]
    log = os.path.join(LOG_DIR, f'NB96_{name}.log')
    print(f'\n=== {name}: weights={weights} clamp={clamp} match={match} ===\n', flush=True)
    rc = run_streaming(cmd, log_path=log, check=False)
    print(f'{name} rc={rc}')
    return rc
print('[ready] run_row defined')


[ready] run_row defined


### Cell 4: Row A - control (cut5x, clamp 100, hungarian) = NB94 baseline

In [4]:
run_row('abl_A_cut5x_clamp_hung', weights='cut5x', clamp='100', match='hungarian')



=== abl_A_cut5x_clamp_hung: weights=cut5x clamp=100 match=hungarian ===

[run_streaming] command: /usr/bin/python3 -u stage2/rmt_ppad_migration/P8_train/scripts/train_lane_only.py --mode full --name abl_A_cut5x_clamp_hung --project /content/runs --model-yaml /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage2/rmt_ppad_migration/vendor/RMT-PPAD/ultralytics/cfg/models/mt-detr/rtdetr-l_bdd_clr_lane.yaml --data-yaml /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage2/rmt_ppad_migration/vendor/RMT-PPAD/ultralytics/cfg/datasets/BDD_lane_only_2k.yaml --device 0 --workers 8 --save-period 50 --batch 32 --epochs 3 --lr0 4e-4 --lane-match hungarian --lane-weights cut5x --diff-clamp 100
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/NB96_abl_A_cut5x_clamp_hung.log
[train_lane_only] LANE_MATCH=hungarian LANE_WEIGHT_PRESET=cut5x LANE_DIFF_CLAMP=100
[train_lane_only] mode=full  epochs=3  batch=32  lr0=0.0004
  model  = /content/drive/MyDrive/EcoCAR/yolop_

0

### Cell 5: Row B - clrkd weights, NO clamp, hungarian

In [5]:
run_row('abl_B_clrkd_noclamp_hung', weights='clrkd', clamp='none', match='hungarian')



=== abl_B_clrkd_noclamp_hung: weights=clrkd clamp=none match=hungarian ===

[run_streaming] command: /usr/bin/python3 -u stage2/rmt_ppad_migration/P8_train/scripts/train_lane_only.py --mode full --name abl_B_clrkd_noclamp_hung --project /content/runs --model-yaml /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage2/rmt_ppad_migration/vendor/RMT-PPAD/ultralytics/cfg/models/mt-detr/rtdetr-l_bdd_clr_lane.yaml --data-yaml /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage2/rmt_ppad_migration/vendor/RMT-PPAD/ultralytics/cfg/datasets/BDD_lane_only_2k.yaml --device 0 --workers 8 --save-period 50 --batch 32 --epochs 3 --lr0 4e-4 --lane-match hungarian --lane-weights clrkd --diff-clamp none
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/NB96_abl_B_clrkd_noclamp_hung.log
[train_lane_only] LANE_MATCH=hungarian LANE_WEIGHT_PRESET=clrkd LANE_DIFF_CLAMP=none
[train_lane_only] mode=full  epochs=3  batch=32  lr0=0.0004
  model  = /content/drive/MyDrive/EcoC

0

### Cell 6: Row C - clrkd weights, no clamp, dynamic_k

In [6]:
run_row('abl_C_clrkd_noclamp_dynk', weights='clrkd', clamp='none', match='dynamic_k')



=== abl_C_clrkd_noclamp_dynk: weights=clrkd clamp=none match=dynamic_k ===

[run_streaming] command: /usr/bin/python3 -u stage2/rmt_ppad_migration/P8_train/scripts/train_lane_only.py --mode full --name abl_C_clrkd_noclamp_dynk --project /content/runs --model-yaml /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage2/rmt_ppad_migration/vendor/RMT-PPAD/ultralytics/cfg/models/mt-detr/rtdetr-l_bdd_clr_lane.yaml --data-yaml /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage2/rmt_ppad_migration/vendor/RMT-PPAD/ultralytics/cfg/datasets/BDD_lane_only_2k.yaml --device 0 --workers 8 --save-period 50 --batch 32 --epochs 3 --lr0 4e-4 --lane-match dynamic_k --lane-weights clrkd --diff-clamp none
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/NB96_abl_C_clrkd_noclamp_dynk.log
[train_lane_only] LANE_MATCH=dynamic_k LANE_WEIGHT_PRESET=clrkd LANE_DIFF_CLAMP=none
[train_lane_only] mode=full  epochs=3  batch=32  lr0=0.0004
  model  = /content/drive/MyDrive/EcoC

0

### Cell 7: Aggregate - which lever unfroze the geometry?

Reads each row's `full_train.log`, extracts the IoU(lane) trajectory and
the first/last `[lane-geom]` means, and prints a comparison + the winner.


In [7]:
import re
from pathlib import Path
ROWS = ['abl_A_cut5x_clamp_hung', 'abl_B_clrkd_noclamp_hung', 'abl_C_clrkd_noclamp_dynk']
def parse(name):
    for base in (Path('/content/runs')/name/'full_train.log',
                 Path('/content/drive/MyDrive/EcoCAR/training_runs/checkpoints')/name/'full_train.log'):
        if base.exists():
            txt = base.read_text(encoding='utf-8', errors='replace'); break
    else:
        return None
    ious = [float(m.group(6)) for l in txt.splitlines()
            if (m := re.match(r'\s*(\d+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)', l))]
    geoms = [tuple(float(x) for x in m.groups()) for l in txt.splitlines()
             if (m := re.search(r'start_x mean=([-\d.]+).*theta mean=([-\d.]+).*length mean=([-\d.]+)', l))]
    iou_rng = (min(ious), max(ious)) if ious else (0, 0)
    geom_moved = len(geoms) >= 2 and any(abs(a-b) > 1e-4 for a, b in zip(geoms[0], geoms[-1]))
    return {'iou_min': iou_rng[0], 'iou_max': iou_rng[1], 'iou_moved': iou_rng[1]-iou_rng[0] > 0.01,
            'geom_moved': geom_moved, 'n_ep': len(ious)}

print(f'{"row":28s} {"IoU min..max":18s} {"IoU moved":10s} {"geom moved":10s}')
best = None
for r in ROWS:
    d = parse(r)
    if d is None: print(f'{r:28s} (no log - run its cell)'); continue
    print(f'{r:28s} {d["iou_min"]:.4f}..{d["iou_max"]:.4f}      {str(d["iou_moved"]):10s} {str(d["geom_moved"]):10s}')
    if d['iou_moved'] and (best is None): best = r
print('\n================ VERDICT ================')
if best:
    print(f'  WINNER: {best} - carry its (weights, clamp, match) into the NB97+ full run.')
else:
    print('  No row moved IoU at 2k/3ep. If NB95 overfit DID move, try more epochs or')
    print('  a lane-only (freeze-detection) row; else the metric (Fault B) is masking it.')


row                          IoU min..max       IoU moved  geom moved
abl_A_cut5x_clamp_hung       0.0034..0.0180      True       False     
abl_B_clrkd_noclamp_hung     0.0049..0.0203      True       False     
abl_C_clrkd_noclamp_dynk     0.0044..0.0193      True       False     

================ VERDICT ================
  WINNER: abl_A_cut5x_clamp_hung - carry its (weights, clamp, match) into the NB97+ full run.
